# Phase 3 Part 1: LightGCN on All 5 Datasets
**CSC14114 - Big Data Applications | Official LightGCN GitHub repo workflow**

This notebook is **Phase 3 - Part 1** only.

## Phase 3 split
1. Part 1: run **LightGCN** on all 5 datasets
2. Part 2: run **SGL** on all 5 datasets
3. Part 3: run **SimGCL** on all 5 datasets and compare all 3 models

## What this notebook does
- uses the official PyTorch LightGCN repository: `gusye1234/LightGCN-PyTorch`
- downloads benchmark datasets from official / cited sources
- uses your **Phase 2 Last.fm dataset** as the 5th dataset
- runs **3 seeds per dataset** and reports mean/std
- saves logs, metrics, runtime, and convergence curves

## Datasets used in Part 1
- `ml-1m`
- `gowalla`
- `yelp2018`
- `amazon-book`
- `lastfm_phase2`

## Kaggle setup
- Turn **Internet = ON**
- Turn **GPU = ON**
- Upload this notebook
- Add a Kaggle input dataset that contains either:
  - `lastfm_phase2_processed.zip`, or
  - extracted files `interactions.csv`, `users.csv`, `items.csv`, `manifest.json`
- Then click **Run All** and **Save Version**


In [ ]:
import ast
import base64
import io
import json
import os
import random
import re
import shutil
import subprocess
import sys
import time
import zipfile
from collections import defaultdict
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch

ROOT = Path('/kaggle/working/phase3_part1_lightgcn') if Path('/kaggle').exists() else Path('phase3_part1_lightgcn')
WORK = ROOT / 'workspace'
REPO_DIR = WORK / 'LightGCN-PyTorch'
RESULTS_DIR = ROOT / 'results'
FIG_DIR = RESULTS_DIR / 'figures'
INPUT_DIR = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path.cwd()
for p in [WORK, RESULTS_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Workspace:', ROOT.resolve())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
LASTFM_REQUIRED_FILES = ['interactions.csv', 'users.csv', 'items.csv', 'manifest.json']


def find_lastfm_phase2_source(input_dir: Path):
    zip_candidates = sorted(input_dir.rglob('*.zip'))
    for zip_path in zip_candidates:
        try:
            with zipfile.ZipFile(zip_path) as zf:
                names = {Path(name).name for name in zf.namelist() if not name.endswith('/')}
            if all(name in names for name in LASTFM_REQUIRED_FILES):
                return ('zip', zip_path)
        except zipfile.BadZipFile:
            continue

    seen = set()
    for manifest_path in sorted(input_dir.rglob('manifest.json')):
        folder = manifest_path.parent
        folder_key = str(folder.resolve())
        if folder_key in seen:
            continue
        seen.add(folder_key)
        if all((folder / name).exists() for name in LASTFM_REQUIRED_FILES):
            return ('dir', folder)

    raise FileNotFoundError(
        'Could not find Last.fm Phase 2 input data under /kaggle/input. '
        'Attach a Kaggle dataset containing lastfm_phase2_processed.zip or the extracted processed CSV/JSON files.'
    )


def materialize_lastfm_phase2_input() -> Path:
    target_dir = WORK / 'lastfm_phase2_input'
    if target_dir.exists() and all((target_dir / name).exists() for name in LASTFM_REQUIRED_FILES):
        return target_dir

    if target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    source_kind, source_path = find_lastfm_phase2_source(INPUT_DIR)
    if source_kind == 'zip':
        with zipfile.ZipFile(source_path) as zf:
            zf.extractall(target_dir)
    else:
        for name in LASTFM_REQUIRED_FILES:
            shutil.copy2(source_path / name, target_dir / name)

    return target_dir


LASTFM_PHASE2_INPUT_DIR = materialize_lastfm_phase2_input()
print(f'Last.fm Phase 2 input ready at: {LASTFM_PHASE2_INPUT_DIR}')


In [ ]:
REPO_URL = 'https://github.com/gusye1234/LightGCN-PyTorch.git'
SEEDS = [42, 52, 62]
TOPKS = [10, 20]

DATASET_RUN_CONFIG = {
    'ml-1m': {'epochs': 200, 'bpr_batch': 2048, 'recdim': 64, 'layer': 3, 'lr': 0.001, 'decay': 1e-4, 'testbatch': 100},
    'gowalla': {'epochs': 200, 'bpr_batch': 2048, 'recdim': 64, 'layer': 3, 'lr': 0.001, 'decay': 1e-4, 'testbatch': 100},
    'yelp2018': {'epochs': 200, 'bpr_batch': 2048, 'recdim': 64, 'layer': 3, 'lr': 0.001, 'decay': 1e-4, 'testbatch': 100},
    'amazon-book': {'epochs': 200, 'bpr_batch': 2048, 'recdim': 64, 'layer': 3, 'lr': 0.001, 'decay': 1e-4, 'testbatch': 100},
    'lastfm_phase2': {'epochs': 200, 'bpr_batch': 2048, 'recdim': 64, 'layer': 3, 'lr': 0.001, 'decay': 1e-4, 'testbatch': 100},
}

RAW_SPLIT_URLS = {
    'gowalla': {
        'train': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/gowalla/train.txt',
        'test': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/gowalla/test.txt',
    },
    'yelp2018': {
        'train': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/yelp2018/train.txt',
        'test': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/yelp2018/test.txt',
    },
    'amazon-book': {
        'train': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/amazon-book/train.txt',
        'test': 'https://raw.githubusercontent.com/gusye1234/LightGCN-PyTorch/master/data/amazon-book/test.txt',
    },
}

MOVIELENS_URL = 'https://files.grouplens.org/datasets/movielens/ml-1m.zip'


In [ ]:
def run_cmd(cmd, cwd=None, capture_output=False):
    print('$', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=cwd, check=True, text=True, capture_output=capture_output)


def download_file(url: str, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        print(f'[skip] {dest.name}')
        return dest
    print(f'[download] {url}')
    response = requests.get(url, stream=True, timeout=300)
    response.raise_for_status()
    with dest.open('wb') as f:
        for chunk in response.iter_content(1024 * 1024):
            if chunk:
                f.write(chunk)
    return dest


def write_repo_split(folder: Path, train_map: dict, test_map: dict):
    folder.mkdir(parents=True, exist_ok=True)
    with (folder / 'train.txt').open('w', encoding='utf-8') as f:
        for uid in sorted(train_map):
            items = train_map[uid]
            if items:
                f.write(str(uid) + ' ' + ' '.join(str(i) for i in items) + '\n')
    with (folder / 'test.txt').open('w', encoding='utf-8') as f:
        for uid in sorted(test_map):
            items = test_map[uid]
            if items:
                f.write(str(uid) + ' ' + ' '.join(str(i) for i in items) + '\n')


def build_leave_one_out_maps(df: pd.DataFrame, user_col: str, item_col: str, time_col: str):
    work = df[[user_col, item_col, time_col]].copy()
    work = work.sort_values([user_col, time_col, item_col], kind='mergesort')
    users = sorted(work[user_col].unique().tolist())
    items = sorted(work[item_col].unique().tolist())
    u_map = {u: i for i, u in enumerate(users)}
    i_map = {it: i for i, it in enumerate(items)}
    work['uid'] = work[user_col].map(u_map).astype(int)
    work['iid'] = work[item_col].map(i_map).astype(int)

    train_map = {}
    test_map = {}
    for uid, group in work.groupby('uid', sort=True):
        item_list = group['iid'].tolist()
        if len(item_list) < 2:
            continue
        train_map[int(uid)] = [int(x) for x in item_list[:-1]]
        test_map[int(uid)] = [int(item_list[-1])]
    return train_map, test_map


def clone_and_patch_repo():
    if not REPO_DIR.exists():
        run_cmd(['git', 'clone', REPO_URL, str(REPO_DIR)])
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'tensorboardX', 'cppimport', 'pybind11'])

    world_path = REPO_DIR / 'code' / 'world.py'
    register_path = REPO_DIR / 'code' / 'register.py'

    world_text = world_path.read_text(encoding='utf-8')
    world_text = world_text.replace(
        "all_dataset = ['lastfm', 'gowalla', 'yelp2018', 'amazon-book']",
        "all_dataset = ['lastfm', 'gowalla', 'yelp2018', 'amazon-book', 'ml-1m', 'lastfm_phase2']"
    )
    world_path.write_text(world_text, encoding='utf-8')

    register_text = '\n'.join([
        'import world',
        'import dataloader',
        'import model',
        'import utils',
        'from pprint import pprint',
        '',
        "if world.dataset == 'lastfm':",
        '    dataset = dataloader.LastFM()',
        'else:',
        "    dataset = dataloader.Loader(path='../data/' + world.dataset)",
        '',
        "print('===========config================')",
        'pprint(world.config)',
        "print('cores for test:', world.CORES)",
        "print('comment:', world.comment)",
        "print('tensorboard:', world.tensorboard)",
        "print('LOAD:', world.LOAD)",
        "print('Weight path:', world.PATH)",
        "print('Test Topks:', world.topks)",
        "print('using bpr loss')",
        "print('===========end===================')",
        'MODELS = {',
        "    'mf': model.PureMF,",
        "    'lgn': model.LightGCN,",
        '}',
        '',
    ])
    register_path.write_text(register_text, encoding='utf-8')
    print('Official LightGCN repo is ready.')


In [ ]:
def prepare_standard_repo_splits():
    data_root = REPO_DIR / 'data'
    for name, urls in RAW_SPLIT_URLS.items():
        folder = data_root / name
        folder.mkdir(parents=True, exist_ok=True)
        download_file(urls['train'], folder / 'train.txt')
        download_file(urls['test'], folder / 'test.txt')


def prepare_ml_1m_split():
    folder = REPO_DIR / 'data' / 'ml-1m'
    if (folder / 'train.txt').exists() and (folder / 'test.txt').exists():
        print('ml-1m already prepared')
        return
    zip_path = download_file(MOVIELENS_URL, WORK / 'ml-1m.zip')
    extract_dir = WORK / 'ml-1m_extracted'
    if not extract_dir.exists():
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)
    ratings_path = extract_dir / 'ml-1m' / 'ratings.dat'
    df = pd.read_csv(
        ratings_path,
        sep='::',
        header=None,
        names=['user_id', 'item_id', 'rating', 'timestamp'],
        engine='python',
        encoding='latin-1',
    )
    df = df[['user_id', 'item_id', 'timestamp']].drop_duplicates(['user_id', 'item_id'])
    train_map, test_map = build_leave_one_out_maps(df, 'user_id', 'item_id', 'timestamp')
    write_repo_split(folder, train_map, test_map)
    print('Prepared ml-1m')


def prepare_lastfm_phase2_split():
    folder = REPO_DIR / 'data' / 'lastfm_phase2'
    if (folder / 'train.txt').exists() and (folder / 'test.txt').exists():
        print('lastfm_phase2 already prepared')
        return
    df = pd.read_csv(LASTFM_PHASE2_INPUT_DIR / 'interactions.csv')
    df['timestamp'] = pd.to_datetime(df['crawl_timestamp_utc'], utc=True, errors='coerce')
    df = df.dropna(subset=['timestamp']).copy()
    df['timestamp'] = df['timestamp'].astype('int64') // 10**9
    df = df[['user_id', 'item_id', 'timestamp']].drop_duplicates(['user_id', 'item_id'])
    train_map, test_map = build_leave_one_out_maps(df, 'user_id', 'item_id', 'timestamp')
    write_repo_split(folder, train_map, test_map)
    print('Prepared lastfm_phase2')


clone_and_patch_repo()
prepare_standard_repo_splits()
prepare_ml_1m_split()
prepare_lastfm_phase2_split()

print('Prepared datasets:')
for p in sorted((REPO_DIR / 'data').iterdir()):
    if p.is_dir():
        print('-', p.name)


In [ ]:
def parse_metric_history(log_text: str):
    pattern = re.compile(r"\{'precision': array\(\[(.*?)\]\), 'recall': array\(\[(.*?)\]\), 'ndcg': array\(\[(.*?)\]\)\}", re.S)
    rows = []
    for idx, match in enumerate(pattern.findall(log_text)):
        prec = [float(x.strip()) for x in match[0].split(',') if x.strip()]
        rec = [float(x.strip()) for x in match[1].split(',') if x.strip()]
        ndcg = [float(x.strip()) for x in match[2].split(',') if x.strip()]
        epoch = idx * 10
        row = {'eval_epoch': epoch}
        for i, k in enumerate(TOPKS):
            row[f'precision@{k}'] = prec[i]
            row[f'recall@{k}'] = rec[i]
            row[f'ndcg@{k}'] = ndcg[i]
        rows.append(row)
    return rows
def run_one(dataset: str, seed: int):
    cfg = DATASET_RUN_CONFIG[dataset]
    run_dir = RESULTS_DIR / dataset / f'seed_{seed}'
    run_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir = run_dir / 'checkpoints'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, 'main.py',
        f'--dataset={dataset}',
        f'--seed={seed}',
        f"--epochs={cfg['epochs']}",
        f"--bpr_batch={cfg['bpr_batch']}",
        f"--recdim={cfg['recdim']}",
        f"--layer={cfg['layer']}",
        f"--lr={cfg['lr']}",
        f"--decay={cfg['decay']}",
        f"--testbatch={cfg['testbatch']}",
        '--tensorboard=0',
        '--multicore=0',
        '--model=lgn',
        f'--topks={json.dumps(TOPKS)}',
        f'--path={ckpt_dir.as_posix()}',
        f'--comment={dataset}-seed-{seed}',
    ]
    print('=' * 90)
    print(f'Running official LightGCN repo: dataset={dataset}, seed={seed}')
    print('=' * 90)
    start = time.time()
    proc = subprocess.run(cmd, cwd=REPO_DIR / 'code', text=True, capture_output=True)
    elapsed = round(time.time() - start, 2)
    log_text = proc.stdout + '\n' + proc.stderr
    log_path = run_dir / 'train.log'
    log_path.write_text(log_text, encoding='utf-8')
    if proc.returncode != 0:
        raise RuntimeError(f'LightGCN failed for dataset={dataset}, seed={seed}. Check {log_path}.')
    history = parse_metric_history(log_text)
    history_df = pd.DataFrame(history)
    history_df.to_csv(run_dir / 'history.csv', index=False)
    final = history[-1] if history else {}
    result = {
        'dataset': dataset,
        'seed': seed,
        'elapsed_seconds': elapsed,
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
        **final,
    }
    (run_dir / 'metrics.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    return result


## Run all Phase 3 Part 1 experiments
This runs **5 datasets x 3 seeds = 15 runs** using the official LightGCN repo.


In [ ]:
all_results = []
for dataset in ['ml-1m', 'gowalla', 'yelp2018', 'amazon-book', 'lastfm_phase2']:
    for seed in SEEDS:
        all_results.append(run_one(dataset, seed))

results_df = pd.DataFrame(all_results)
results_df.to_csv(RESULTS_DIR / 'all_runs.csv', index=False)
display(results_df)


In [ ]:
metric_cols = [c for c in results_df.columns if '@' in c] + ['elapsed_seconds']
summary_rows = []
for dataset, group in results_df.groupby('dataset'):
    row = {'dataset': dataset, 'runs': int(len(group))}
    for col in metric_cols:
        row[f'{col}_mean'] = float(group[col].mean())
        row[f'{col}_std'] = float(group[col].std(ddof=0))
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).sort_values('dataset').reset_index(drop=True)
summary_df.to_csv(RESULTS_DIR / 'summary.csv', index=False)
display(summary_df)


In [ ]:
def read_repo_split_stats(dataset: str):
    folder = REPO_DIR / 'data' / dataset

    def parse_split(path: Path):
        users = set()
        items = set()
        interactions = 0
        with path.open('r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                uid = int(parts[0])
                item_list = [int(x) for x in parts[1:]]
                if not item_list:
                    continue
                users.add(uid)
                items.update(item_list)
                interactions += len(item_list)
        return users, items, interactions

    train_users, train_items, train_interactions = parse_split(folder / 'train.txt')
    test_users, test_items, test_interactions = parse_split(folder / 'test.txt')
    all_users = train_users | test_users
    all_items = train_items | test_items
    total_interactions = train_interactions + test_interactions
    density = total_interactions / (len(all_users) * len(all_items)) if all_users and all_items else 0.0

    return {
        'dataset': dataset,
        'users': len(all_users),
        'items': len(all_items),
        'train_interactions': train_interactions,
        'test_interactions': test_interactions,
        'total_interactions': total_interactions,
        'density': density,
    }


dataset_stats_df = pd.DataFrame(
    [read_repo_split_stats(dataset) for dataset in ['ml-1m', 'gowalla', 'yelp2018', 'amazon-book', 'lastfm_phase2']]
).sort_values('dataset').reset_index(drop=True)
dataset_stats_df.to_csv(RESULTS_DIR / 'dataset_characteristics.csv', index=False)

analysis_df = summary_df.merge(dataset_stats_df, on='dataset', how='left')
analysis_df.to_csv(RESULTS_DIR / 'lightgcn_phase3_part1_analysis.csv', index=False)

display(dataset_stats_df)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
plot_specs = [
    ('recall@20_mean', 'Recall@20', '#3366cc'),
    ('ndcg@20_mean', 'NDCG@20', '#dc3912'),
    ('elapsed_seconds_mean', 'Runtime (s)', '#ff9900'),
    ('density', 'Density', '#109618'),
]

for ax, (col, title, color) in zip(axes.flatten(), plot_specs):
    bars = ax.bar(analysis_df['dataset'], analysis_df[col], color=color)
    ax.set_title(title, fontweight='bold')
    ax.tick_params(axis='x', rotation=20)
    for bar, value in zip(bars, analysis_df[col]):
        label = f'{value:.4f}' if value < 1000 else f'{value:,.0f}'
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), label, ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / 'analysis_overview.png', dpi=150, bbox_inches='tight')
plt.show()

corr_cols = ['users', 'items', 'total_interactions', 'density', 'recall@20_mean', 'ndcg@20_mean']
print('Correlation summary:')
display(analysis_df[corr_cols].corr(numeric_only=True))


In [ ]:
for history_path in sorted(RESULTS_DIR.glob('*/*/history.csv')):
    history_df = pd.read_csv(history_path)
    if history_df.empty:
        continue
    dataset = history_path.parent.parent.name
    seed = history_path.parent.name
    plt.figure(figsize=(7, 4))
    plt.plot(history_df['eval_epoch'], history_df['recall@20'], label='recall@20')
    plt.plot(history_df['eval_epoch'], history_df['ndcg@20'], label='ndcg@20')
    plt.xlabel('Evaluation Epoch')
    plt.title(f'{dataset} - {seed}')
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'{dataset}_{seed}_convergence.png', dpi=150, bbox_inches='tight')
    plt.close()

run_metadata = {
    'repo_url': REPO_URL,
    'datasets': ['ml-1m', 'gowalla', 'yelp2018', 'amazon-book', 'lastfm_phase2'],
    'seeds': SEEDS,
    'topks': TOPKS,
    'torch_version': torch.__version__,
    'cuda_available': bool(torch.cuda.is_available()),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'lastfm_input_dir': str(LASTFM_PHASE2_INPUT_DIR),
}
(RESULTS_DIR / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2), encoding='utf-8')

archive_base = ROOT / 'phase3_part1_lightgcn_results'
archive_zip = archive_base.with_suffix('.zip')
if archive_zip.exists():
    archive_zip.unlink()
shutil.make_archive(str(archive_base), 'zip', RESULTS_DIR)
print(f'Results archive: {archive_zip}')
print('Finished Phase 3 Part 1.')
